# Task 04: Supervised Classification / Regression Modeling & Tuning

**Project:** Breast Cancer Classification using Machine Learning  
**Objective:** Train and compare at least four models, tune hyperparameters with `GridSearchCV` and stratified 5-fold cross-validation, evaluate classification metrics, and save the champion model.

> This is a binary **classification** project. The supplied scikit-learn Wisconsin Diagnostic Breast Cancer dataset is used so the notebook is reproducible without downloading a dataset. Target labels: malignant and benign (encoded by scikit-learn as 0 and 1).

## 1. Imports and dataset


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)

RANDOM_STATE = 42
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print("Dataset shape:", X.shape)
print("Target names:", dataset.target_names)
display(X.head())
print("\\nClass distribution:")
display(y.value_counts().rename(index={0: "malignant", 1: "benign"}))


## 2. Train-test split

A stratified split preserves the target-class proportions in both partitions. The test set is held out until final evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))


## 3. Leak-free preprocessing and model definitions

Imputation and scaling are inside each pipeline. During cross-validation, these transformations are fitted only on each training fold, preventing information from validation folds from leaking into preprocessing.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Logistic Regression": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
        ]),
        {"model__C": [0.1, 1.0, 10.0]}
    ),
    "Random Forest": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"))
        ]),
        {"model__n_estimators": [150, 300], "model__max_depth": [None, 5, 10]}
    ),
    "Gradient Boosting": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
        ]),
        {"model__n_estimators": [100, 150], "model__learning_rate": [0.05, 0.1]}
    ),
    "Extra Trees": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(random_state=RANDOM_STATE, class_weight="balanced"))
        ]),
        {"model__n_estimators": [150, 300], "model__max_depth": [None, 10]}
    )
}
print("Models to tune:", list(models))


## 4. Hyperparameter optimization

Each model is tuned with `GridSearchCV`, using ROC-AUC as the selection metric and stratified 5-fold cross-validation.


In [ ]:
searches = {}
comparison_rows = []

for model_name, (pipeline, param_grid) in models.items():
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True
    )
    search.fit(X_train, y_train)
    searches[model_name] = search

    test_pred = search.predict(X_test)
    test_prob = search.predict_proba(X_test)[:, 1]
    comparison_rows.append({
        "Model": model_name,
        "Best CV ROC-AUC": search.best_score_,
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred),
        "Recall": recall_score(y_test, test_pred),
        "F1-Score": f1_score(y_test, test_pred),
        "Test ROC-AUC": roc_auc_score(y_test, test_prob),
        "Best Parameters": search.best_params_
    })

comparison = pd.DataFrame(comparison_rows).sort_values(
    "Best CV ROC-AUC", ascending=False
).reset_index(drop=True)
display(comparison)


## 5. Select the champion model

The champion is selected using the highest mean cross-validation ROC-AUC, not the test score. The test set remains an unbiased final check.


In [ ]:
champion_name = comparison.loc[0, "Model"]
champion_search = searches[champion_name]
champion_model = champion_search.best_estimator_

print("Champion model:", champion_name)
print("Best CV ROC-AUC:", round(champion_search.best_score_, 4))
print("Best parameters:", champion_search.best_params_)


## 6. Final evaluation: precision, recall, F1, ROC-AUC and confusion matrix


In [ ]:
y_pred = champion_model.predict(X_test)
y_prob = champion_model.predict_proba(X_test)[:, 1]

print("Classification report:")
print(classification_report(y_test, y_pred, target_names=dataset.target_names))
print("Test ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=dataset.target_names, cmap="Blues"
)
plt.title(f"Confusion Matrix - {champion_name}")
plt.tight_layout()
plt.show()


## 7. ROC-AUC curves for all tuned models


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for model_name, search in searches.items():
    RocCurveDisplay.from_estimator(
        search.best_estimator_, X_test, y_test, ax=ax, name=model_name
    )
ax.plot([0, 1], [0, 1], "k--", label="Random baseline")
ax.set_title("ROC Curves - Tuned Model Comparison")
plt.tight_layout()
plt.show()


## 8. Save deliverables

The final pipeline includes preprocessing and the trained estimator. Save the champion model and comparison table for reproducibility and later inference.


In [ ]:
joblib.dump(champion_model, "champion_model.pkl")
comparison.to_csv("model_comparison.csv", index=False)
print("Saved champion_model.pkl and model_comparison.csv")


## 9. Example prediction with the saved model


In [ ]:
loaded_model = joblib.load("champion_model.pkl")
sample = X_test.iloc[[0]]
predicted_class = loaded_model.predict(sample)[0]
predicted_probability = loaded_model.predict_proba(sample)[0, 1]
print("Predicted class:", dataset.target_names[predicted_class])
print("Probability of benign class:", round(predicted_probability, 4))


## 10. Conclusion

Four distinct classifiers were trained and tuned using stratified 5-fold cross-validation. Preprocessing was kept inside the pipelines to avoid leakage. The champion model was selected using cross-validation ROC-AUC and evaluated once on the held-out test set. The notebook exports the serialized end-to-end pipeline and model comparison table.

**Important:** This is an educational machine-learning exercise, not a medical diagnostic tool.
